# SatMesh — Generate 40-Region Dataset + Train Multiclass SegFormer

**Run order:** Save Version → Save & Run All (background)

### Before running
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. No dataset attachment needed — this notebook generates data from scratch

**Expected runtime:** ~40 min data generation + ~4 hrs training = ~5 hrs total

In [ ]:
# 1. Dependencies
!pip -q install "segmentation-models-pytorch>=0.4" "albumentations>=2.0" \
                 osmnx shapely rasterio pyproj pyyaml
import torch
print('torch', torch.__version__,
      '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.device_count(), 'GPU(s)',
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 2. Clone SatMesh repo
import os, sys
REPO = '/kaggle/working/SatMesh'
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/SahilKumar75/SatMesh.git {REPO}
else:
    !git -C {REPO} pull --ff-only
sys.path.insert(0, REPO)
os.chdir(REPO)
print('Repo ready at', REPO)

In [ ]:
# 3. Generate 40-region multiclass dataset (Bing 0.6m + OSM masks)
# ~40 min — Bing tile fetch + OSM road rasterization + tree/shadow heuristics
DATA_OUT = '/kaggle/working/data/india_multiclass/train'
!python scripts/build_multiclass_dataset.py \
    --regions all \
    --zoom 18 \
    --windows 220 \
    --out {DATA_OUT}

In [ ]:
# 4. Verify dataset
import glob
sats   = glob.glob(DATA_OUT + '/*_sat.jpg')
masks  = glob.glob(DATA_OUT + '/*_mask.png')
print(f'sat tiles : {len(sats)}')
print(f'mask tiles: {len(masks)}')
assert len(sats) > 100, 'Too few tiles — check generation logs above'

# Quick mask class distribution check
import cv2, numpy as np, random
sample = random.sample(masks, min(20, len(masks)))
counts = np.zeros(4)
for p in sample:
    m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    for c in range(4):
        counts[c] += (m == c).sum()
total = counts.sum()
labels = ['bg', 'road', 'tree', 'shadow']
for i, l in enumerate(labels):
    print(f'  class {i} {l:7s}: {counts[i]/total*100:.1f}%')

In [ ]:
# 5. Train multiclass SegFormer-B4 (4-class: bg/road/tree/shadow)
# ~4 hrs on T4 x2 for 40 epochs
CKPT = '/kaggle/working/checkpoints/segformer_v3_multiclass.pth'
!python scripts/train_multiclass.py \
    --data {DATA_OUT} \
    --out  {CKPT} \
    --encoder mit_b4 \
    --epochs 40 \
    --batch  8 \
    --lr     6e-5 \
    --img-size 512

In [ ]:
# 6. Results summary
import os
ckpt_size = os.path.getsize(CKPT) / 1e6
state_path = CKPT.replace('.pth', '.training_state.pt')
print(f'Checkpoint : {CKPT}  ({ckpt_size:.0f} MB)')
print(f'State file : {state_path}  (exists={os.path.exists(state_path)})')
print()
print('Download both files from Output tab.')
print('Commit to repo: checkpoints/segformer_v3_multiclass.pth')

In [ ]:
# 7. (Optional) Save dataset as Kaggle dataset for reuse
# Uncomment to package tiles — useful if you want to skip re-generation next time
# import shutil
# shutil.make_archive('/kaggle/working/india_multiclass_40regions', 'zip', DATA_OUT)
# print('Zipped to /kaggle/working/india_multiclass_40regions.zip')
# print('Upload via: Kaggle → Datasets → New → upload that zip')